# Long-Cash Mean Reversion

Tests a long/cash mean-reversion rule using daily historical prices.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
from _market_data import download_price_records
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")


class StrategyConfig:
    """
    Configuration class for Larry Connors MDU/MDD Strategy
    All parameters are exposed and configurable by the user
    """

    TICKERS = ["SPY", "XLK"]
    BENCHMARK_TICKER = "SPY"
    END_DATE = "2024-12-31"
    START_DATE = "2019-12-31"
    MA_LONG_PERIOD = 200
    MA_SHORT_PERIOD = 5
    LOOKBACK_DAYS = 5
    MIN_DOWN_DAYS = 4
    INITIAL_CAPITAL = 100000
    COMPOUNDING_MODE = True
    LEVERAGE = 1.0
    COMMISSION_PCT = 0.001
    TRAIN_YEARS = 2
    VALIDATION_YEARS = 1
    REOPTIMIZE_FREQ = "yearly"
    RISK_FREE_RATE = 0.02
    TARGET_CONFIDENCE = 0.95
    FIGURE_SIZE = (14, 8)
    SAVE_FIGURES = True
    FIGURE_DPI = 300


def download_fmp_data(ticker, start_date, end_date):
    """
    Download historical OHLCV data from Yahoo Finance.

    CRITICAL: No synthetic data or fallback - returns None if download fails.
    This ensures we only use real market data for backtesting.

    Parameters:
    -----------
    ticker : str
        Stock/ETF ticker symbol
    start_date : str
        Start date in 'YYYY-MM-DD' format
    end_date : str
        End date in 'YYYY-MM-DD' format
        Yahoo Finance key

    Returns:
    --------
    pd.DataFrame or None
        DataFrame with columns: ['date', 'open', 'high', 'low', 'close', 'volume']
        Returns None if download fails
    """
    try:
        response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
        data = response_prices
        if "historical" not in data or len(data["historical"]) == 0:
            print(f"WARNING: No data returned for {ticker}")
            return None
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)
        df = df[["date", "open", "high", "low", "close", "volume"]]
        print(
            f"✓ Downloaded {len(df)} bars for {ticker} ({df['date'].min().date()} to {df['date'].max().date()})"
        )
        return df
    except Exception as e:
        print(f"ERROR downloading {ticker}: {str(e)}")
        return None


def load_all_data(config):
    """
    Load data for all tickers and benchmark.

    Returns:
    --------
    dict
        Dictionary with ticker as key and DataFrame as value
    """
    print("\n" + "=" * 80)
    print("DATA DOWNLOAD MODULE")
    print("=" * 80)
    all_data = {}
    valid_tickers = [t for t in config.TICKERS if t and len(t) > 0]
    for ticker in valid_tickers + [config.BENCHMARK_TICKER]:
        df = download_fmp_data(ticker, config.START_DATE, config.END_DATE)
        if df is not None:
            all_data[ticker] = df
        else:
            print(f"FAILED: {ticker} excluded from analysis")
    print("=" * 80)
    print(f"Successfully loaded {len(all_data)} instruments: {list(all_data.keys())}\n")
    return all_data


def calculate_features(df, config):
    """
    Calculate technical indicators for strategy signals.

    CRITICAL: All features use T-1 data to avoid look-ahead bias.
    - Moving averages are calculated on close prices
    - Down days counter looks at prior 5 days only
    - No future data is used in any calculation

    Parameters:
    -----------
    df : pd.DataFrame
        Price data with columns ['date', 'open', 'high', 'low', 'close', 'volume']
    config : StrategyConfig
        Configuration object

    Returns:
    --------
    pd.DataFrame
        Original data with added feature columns
    """
    df = df.copy()
    df["returns"] = df["close"].pct_change()
    df["ma_long"] = (
        df["close"].rolling(window=config.MA_LONG_PERIOD, min_periods=config.MA_LONG_PERIOD).mean()
    )
    df["ma_short"] = (
        df["close"]
        .rolling(window=config.MA_SHORT_PERIOD, min_periods=config.MA_SHORT_PERIOD)
        .mean()
    )
    df["is_down_day"] = (df["close"] < df["close"].shift(1)).astype(int)
    df["down_days_count"] = (
        df["is_down_day"]
        .rolling(window=config.LOOKBACK_DAYS, min_periods=config.LOOKBACK_DAYS)
        .sum()
    )
    df["ma_long_lag"] = df["ma_long"].shift(1)
    df["ma_short_lag"] = df["ma_short"].shift(1)
    df["close_lag"] = df["close"].shift(1)
    df["down_days_count_lag"] = df["down_days_count"].shift(1)
    return df


def generate_signals(df, config):
    """
    Generate buy/sell signals based on Connors MDU/MDD rules.

    ENTRY RULES (all must be true at T-1 close):
    1. Close > 200-day MA (long-term uptrend confirmation)
    2. Close < 5-day MA (short-term pullback)
    3. At least 4 out of last 5 days were down days

    EXIT RULES:
    - Exit when Close > 5-day MA (mean reversion complete)

    NO STOP-LOSS per original Connors strategy

    TIMING:
    - Signal generated at T-1 close
    - Order executed at T open (next day)
    - This prevents look-ahead bias

    Parameters:
    -----------
    df : pd.DataFrame
        Data with features
    config : StrategyConfig
        Configuration object

    Returns:
    --------
    pd.DataFrame
        Data with added signal columns
    """
    df = df.copy()
    df["signal"] = 0
    condition_trend = df["close_lag"] > df["ma_long_lag"]
    condition_pullback = df["close_lag"] < df["ma_short_lag"]
    condition_down_days = df["down_days_count_lag"] >= config.MIN_DOWN_DAYS
    entry_signal = condition_trend & condition_pullback & condition_down_days
    exit_signal = df["close_lag"] >= df["ma_short_lag"]
    position = 0
    for i in range(len(df)):
        if pd.isna(df.loc[i, "ma_long_lag"]) or pd.isna(df.loc[i, "ma_short_lag"]):
            df.loc[i, "signal"] = 0
            position = 0
            continue
        if position == 1 and exit_signal.iloc[i]:
            position = 0
        elif position == 0 and entry_signal.iloc[i]:
            position = 1
        df.loc[i, "signal"] = position
    df["position"] = df["signal"].shift(1).fillna(0)
    return df


def calculate_position_size(portfolio_value, current_capital, config):
    """
    Calculate position size based on sizing methodology.

    CRITICAL: Position sizing methodology to avoid artificial leverage bug

    TWO MODES:
    1. COMPOUNDING_MODE = False (RESEARCH MODE - DEFAULT):
       - Size positions on INITIAL_CAPITAL (fixed notional)
       - Maintains constant dollar exposure throughout backtest
       - Returns reflect pure signal alpha without leverage drift
       - Use for: Alpha research, strategy comparison, market-neutral

    2. COMPOUNDING_MODE = True (PRODUCTION MODE):
       - Size positions on current portfolio value
       - Simulates real fund with reinvestment
       - Returns are geometrically compounded
       - Use for: Investor presentations, AUM projections

    LEVERAGE parameter controls risk level independently of compounding

    Parameters:
    -----------
    portfolio_value : float
        Current portfolio value (used if compounding=True)
    current_capital : float
        Initial capital (used if compounding=False)
    config : StrategyConfig
        Configuration object

    Returns:
    --------
    float
        Position size in dollars
    """
    if config.COMPOUNDING_MODE:
        base_size = portfolio_value
    else:
        base_size = current_capital
    position_size = base_size * config.LEVERAGE
    return position_size


def backtest_strategy(df, config, ticker):
    """
    Execute backtest with realistic execution and cost assumptions.

    EXECUTION TIMING (strict T-1 to T logic):
    - Day T-1 Close: Signals generated using data up to T-1
    - Day T Open: Orders executed at open price
    - Day T Close: Portfolio marked-to-market at close price

    COST MODEL:
    - Entry commission: config.COMMISSION_PCT * position_value
    - Exit commission: config.COMMISSION_PCT * position_value
    - No slippage assumed (conservative for liquid ETFs)

    NO STOP-LOSS per Connors original strategy

    Parameters:
    -----------
    df : pd.DataFrame
        Data with signals
    config : StrategyConfig
        Configuration object
    ticker : str
        Ticker symbol for tracking

    Returns:
    --------
    pd.DataFrame
        Data with performance columns
    """
    df = df.copy()
    portfolio_value = config.INITIAL_CAPITAL
    cash = config.INITIAL_CAPITAL
    shares = 0
    entry_price = 0
    entry_commission = 0
    df["portfolio_value"] = config.INITIAL_CAPITAL
    df["cash"] = config.INITIAL_CAPITAL
    df["shares"] = 0
    df["trade"] = ""
    df["pnl"] = 0
    df["commission"] = 0
    for i in range(1, len(df)):
        current_position = df.loc[i, "position"]
        previous_position = df.loc[i - 1, "position"]
        open_price = df.loc[i, "open"]
        close_price = df.loc[i, "close"]
        if current_position == 1 and previous_position == 0:
            position_size = calculate_position_size(
                portfolio_value=df.loc[i - 1, "portfolio_value"],
                current_capital=config.INITIAL_CAPITAL,
                config=config,
            )
            shares = position_size / open_price
            entry_price = open_price
            entry_commission = position_size * config.COMMISSION_PCT
            cash = cash - shares * open_price - entry_commission
            df.loc[i, "trade"] = "BUY"
            df.loc[i, "commission"] = entry_commission
        elif current_position == 0 and previous_position == 1:
            exit_value = shares * open_price
            exit_commission = exit_value * config.COMMISSION_PCT
            pnl = exit_value - shares * entry_price - entry_commission - exit_commission
            cash = cash + exit_value - exit_commission
            df.loc[i, "trade"] = "SELL"
            df.loc[i, "pnl"] = pnl
            df.loc[i, "commission"] = exit_commission
            shares = 0
            entry_price = 0
            entry_commission = 0
        if shares > 0:
            portfolio_value = cash + shares * close_price
        else:
            portfolio_value = cash
        df.loc[i, "portfolio_value"] = portfolio_value
        df.loc[i, "cash"] = cash
        df.loc[i, "shares"] = shares
    df["strategy_returns"] = df["portfolio_value"].pct_change().fillna(0)
    df["bh_returns"] = df["close"].pct_change().fillna(0)
    df["strategy_cumulative"] = (1 + df["strategy_returns"]).cumprod()
    df["bh_cumulative"] = (1 + df["bh_returns"]).cumprod()
    df["cumulative_max"] = df["portfolio_value"].expanding().max()
    df["drawdown"] = (df["portfolio_value"] - df["cumulative_max"]) / df["cumulative_max"]
    return df


def calculate_performance_metrics(df, config, ticker):
    """
    Calculate comprehensive performance statistics.

    METRICS INCLUDED:
    - Total Return (%)
    - Annualized Return (%)
    - Annualized Volatility (%)
    - Sharpe Ratio (annualized)
    - Sortino Ratio (annualized, using downside deviation)
    - Maximum Drawdown (%)
    - Calmar Ratio (return / max drawdown)
    - Win Rate (%)
    - Profit Factor
    - Average Trade P&L
    - Number of Trades
    - Probabilistic Sharpe Ratio (PSR)
    - Deflated Sharpe Ratio (DSR)

    Parameters:
    -----------
    df : pd.DataFrame
        Backtest results
    config : StrategyConfig
        Configuration object
    ticker : str
        Ticker symbol

    Returns:
    --------
    dict
        Dictionary of performance metrics
    """
    total_return = (df["portfolio_value"].iloc[-1] / config.INITIAL_CAPITAL - 1) * 100
    days = len(df)
    years = days / 252
    annualized_return = (
        (df["portfolio_value"].iloc[-1] / config.INITIAL_CAPITAL) ** (1 / years) - 1
    ) * 100
    volatility = df["strategy_returns"].std() * np.sqrt(252) * 100
    excess_returns = df["strategy_returns"] - config.RISK_FREE_RATE / 252
    sharpe_ratio = (
        excess_returns.mean() / df["strategy_returns"].std() * np.sqrt(252)
        if df["strategy_returns"].std() > 0
        else 0
    )
    negative_returns = df["strategy_returns"][df["strategy_returns"] < 0]
    downside_std = (
        negative_returns.std() if len(negative_returns) > 0 else df["strategy_returns"].std()
    )
    sortino_ratio = excess_returns.mean() / downside_std * np.sqrt(252) if downside_std > 0 else 0
    cumulative = (1 + df["strategy_returns"]).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min() * 100
    calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0
    trades = df[df["trade"].isin(["BUY", "SELL"])]
    num_trades = len(trades[trades["trade"] == "BUY"])
    closed_trades = df[df["trade"] == "SELL"]
    if len(closed_trades) > 0:
        winning_trades = closed_trades[closed_trades["pnl"] > 0]
        losing_trades = closed_trades[closed_trades["pnl"] < 0]
        win_rate = len(winning_trades) / len(closed_trades) * 100
        avg_trade_pnl = closed_trades["pnl"].mean()
        total_wins = winning_trades["pnl"].sum() if len(winning_trades) > 0 else 0
        total_losses = abs(losing_trades["pnl"].sum()) if len(losing_trades) > 0 else 1
        profit_factor = total_wins / total_losses if total_losses > 0 else 0
    else:
        win_rate = 0
        avg_trade_pnl = 0
        profit_factor = 0
    n = len(df)
    skewness = df["strategy_returns"].skew()
    kurtosis = df["strategy_returns"].kurtosis()
    if n > 0 and df["strategy_returns"].std() > 0:
        psr_numerator = (sharpe_ratio - 0) * np.sqrt(n - 1)
        psr_denominator = np.sqrt(
            1 - skewness * sharpe_ratio + (kurtosis - 1) / 4 * sharpe_ratio**2
        )
        psr = stats.norm.cdf(psr_numerator / psr_denominator) if psr_denominator > 0 else 0
    else:
        psr = 0
    num_tests = 10
    dsr_threshold = np.sqrt(2 * np.log(num_tests) / n) if n > 0 else 0
    dsr = sharpe_ratio - dsr_threshold
    metrics = {
        "ticker": ticker,
        "total_return_pct": total_return,
        "annualized_return_pct": annualized_return,
        "volatility_pct": volatility,
        "sharpe_ratio": sharpe_ratio,
        "sortino_ratio": sortino_ratio,
        "max_drawdown_pct": max_drawdown,
        "calmar_ratio": calmar_ratio,
        "win_rate_pct": win_rate,
        "profit_factor": profit_factor,
        "avg_trade_pnl": avg_trade_pnl,
        "num_trades": num_trades,
        "psr": psr,
        "dsr": dsr,
        "final_portfolio_value": df["portfolio_value"].iloc[-1],
    }
    return metrics


def calculate_benchmark_metrics(df, config):
    """
    Calculate buy-and-hold benchmark performance.

    Parameters:
    -----------
    df : pd.DataFrame
        Benchmark price data
    config : StrategyConfig
        Configuration object

    Returns:
    --------
    dict
        Dictionary of benchmark metrics
    """
    df = df.copy()
    df["returns"] = df["close"].pct_change().fillna(0)
    total_return = (df["close"].iloc[-1] / df["close"].iloc[0] - 1) * 100
    days = len(df)
    years = days / 252
    annualized_return = ((df["close"].iloc[-1] / df["close"].iloc[0]) ** (1 / years) - 1) * 100
    volatility = df["returns"].std() * np.sqrt(252) * 100
    excess_returns = df["returns"] - config.RISK_FREE_RATE / 252
    sharpe_ratio = (
        excess_returns.mean() / df["returns"].std() * np.sqrt(252) if df["returns"].std() > 0 else 0
    )
    cumulative = (1 + df["returns"]).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min() * 100
    metrics = {
        "ticker": config.BENCHMARK_TICKER,
        "total_return_pct": total_return,
        "annualized_return_pct": annualized_return,
        "volatility_pct": volatility,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown_pct": max_drawdown,
    }
    return metrics


def walk_forward_analysis(df, config, ticker):
    """
    Perform walk-forward expanding window analysis.

    METHODOLOGY:
    - Expanding window approach (not rolling)
    - Each window includes all prior data
    - Prevents overfitting by testing on unseen future data
    - No reoptimization in baseline (parameters fixed)

    STRUCTURE:
    - Initial train period: 2 years
    - Test on subsequent 1-year periods
    - Repeat until end of data

    This validates that strategy works out-of-sample consistently

    Parameters:
    -----------
    df : pd.DataFrame
        Full dataset with signals
    config : StrategyConfig
        Configuration object
    ticker : str
        Ticker symbol

    Returns:
    --------
    list
        List of dictionaries with per-period results
    """
    df["year"] = df["date"].dt.year
    years = sorted(df["year"].unique())
    if len(years) < 3:
        print(f"WARNING: Insufficient data for walk-forward analysis on {ticker}")
        return []
    results = []
    train_end_year = years[0] + config.TRAIN_YEARS - 1
    test_years = [y for y in years if y > train_end_year]
    for test_year in test_years:
        train_data = df[df["year"] < test_year].copy()
        test_data = df[df["year"] == test_year].copy()
        if len(train_data) < 252 or len(test_data) < 20:
            continue
        test_metrics = calculate_performance_metrics(test_data, config, ticker)
        test_metrics["period"] = f"{test_year}"
        test_metrics["period_type"] = "test"
        test_metrics["train_size"] = len(train_data)
        test_metrics["test_size"] = len(test_data)
        results.append(test_metrics)
    return results


def plot_equity_curve(results_dict, benchmark_df, config):
    """
    Plot equity curves for all strategies vs benchmark.

    Creates a comprehensive equity curve comparison showing portfolio growth
    over time for each ticker strategy and the benchmark.

    Parameters:
    -----------
    results_dict : dict
        Dictionary with ticker as key and backtest DataFrame as value
    benchmark_df : pd.DataFrame
        Benchmark price data
    config : StrategyConfig
        Configuration object
    """
    fig, ax = plt.subplots(figsize=config.FIGURE_SIZE)
    for ticker, df in results_dict.items():
        ax.plot(
            df["date"], df["portfolio_value"], label=f"{ticker} Strategy", linewidth=2, alpha=0.8
        )
    if benchmark_df is not None:
        benchmark_returns = benchmark_df["close"].pct_change().fillna(0)
        benchmark_value = config.INITIAL_CAPITAL * (1 + benchmark_returns).cumprod()
        ax.plot(
            benchmark_df["date"],
            benchmark_value,
            label=f"{config.BENCHMARK_TICKER} (Buy & Hold)",
            linewidth=2,
            linestyle="--",
            color="black",
            alpha=0.6,
        )
    ax.set_xlabel("Date", fontsize=12, fontweight="bold")
    ax.set_ylabel("Portfolio Value ($)", fontsize=12, fontweight="bold")
    ax.set_title("Equity Curve: Strategy Performance vs Benchmark", fontsize=14, fontweight="bold")
    ax.legend(loc="best", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
    plt.tight_layout()
    if config.SAVE_FIGURES:
        filename = f"equity_curve_{config.START_DATE}_{config.END_DATE}.png"
        plt.savefig(filename, dpi=config.FIGURE_DPI, bbox_inches="tight")
        print(f"✓ Saved equity curve to: {filename}")
    plt.show()
    plt.close()


def plot_drawdown(results_dict, config):
    """
    Plot drawdown charts for all strategies.

    Visualizes the peak-to-trough decline for each strategy,
    helping identify periods of high risk and recovery patterns.

    Parameters:
    -----------
    results_dict : dict
        Dictionary with ticker as key and backtest DataFrame as value
    config : StrategyConfig
        Configuration object
    """
    fig, ax = plt.subplots(figsize=config.FIGURE_SIZE)
    for ticker, df in results_dict.items():
        ax.plot(df["date"], df["drawdown"] * 100, label=ticker, linewidth=2, alpha=0.7)
    ax.fill_between(df["date"], 0, df["drawdown"] * 100, alpha=0.1)
    ax.set_xlabel("Date", fontsize=12, fontweight="bold")
    ax.set_ylabel("Drawdown (%)", fontsize=12, fontweight="bold")
    ax.set_title("Strategy Drawdown Over Time", fontsize=14, fontweight="bold")
    ax.legend(loc="best", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color="black", linestyle="-", linewidth=0.8)
    plt.tight_layout()
    if config.SAVE_FIGURES:
        filename = f"drawdown_{config.START_DATE}_{config.END_DATE}.png"
        plt.savefig(filename, dpi=config.FIGURE_DPI, bbox_inches="tight")
        print(f"✓ Saved drawdown chart to: {filename}")
    plt.show()
    plt.close()


def plot_returns_distribution(results_dict, config):
    """
    Plot returns distribution histograms for all strategies.

    Shows the frequency distribution of daily returns, helping assess
    the symmetry and tail risk of the strategy.

    Parameters:
    -----------
    results_dict : dict
        Dictionary with ticker as key and backtest DataFrame as value
    config : StrategyConfig
        Configuration object
    """
    n_strategies = len(results_dict)
    n_cols = min(3, n_strategies)
    n_rows = (n_strategies + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    if n_strategies == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if n_strategies > 1 else [axes]
    for idx, (ticker, df) in enumerate(results_dict.items()):
        ax = axes[idx]
        returns = df["strategy_returns"].dropna() * 100
        ax.hist(returns, bins=50, alpha=0.7, color="steelblue", edgecolor="black")
        ax.axvline(
            returns.mean(),
            color="red",
            linestyle="--",
            linewidth=2,
            label=f"Mean: {returns.mean():.2f}%",
        )
        ax.axvline(
            returns.median(),
            color="green",
            linestyle="--",
            linewidth=2,
            label=f"Median: {returns.median():.2f}%",
        )
        ax.set_xlabel("Daily Returns (%)", fontsize=10, fontweight="bold")
        ax.set_ylabel("Frequency", fontsize=10, fontweight="bold")
        ax.set_title(f"{ticker} - Returns Distribution", fontsize=11, fontweight="bold")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    for idx in range(n_strategies, len(axes)):
        axes[idx].axis("off")
    plt.tight_layout()
    if config.SAVE_FIGURES:
        filename = f"returns_distribution_{config.START_DATE}_{config.END_DATE}.png"
        plt.savefig(filename, dpi=config.FIGURE_DPI, bbox_inches="tight")
        print(f"✓ Saved returns distribution to: {filename}")
    plt.show()
    plt.close()


def plot_monthly_returns_heatmap(df, ticker, config):
    """
    Plot monthly returns heatmap for a single strategy.

    Creates a calendar-style heatmap showing monthly returns,
    making it easy to identify seasonal patterns and consistency.

    Parameters:
    -----------
    df : pd.DataFrame
        Backtest results
    ticker : str
        Ticker symbol
    config : StrategyConfig
        Configuration object
    """
    df_monthly = df.copy()
    df_monthly["year"] = df_monthly["date"].dt.year
    df_monthly["month"] = df_monthly["date"].dt.month
    monthly_returns = (
        df_monthly.groupby(["year", "month"])["portfolio_value"].last().pct_change() * 100
    )
    monthly_returns = monthly_returns.reset_index()
    monthly_returns.columns = ["year", "month", "return"]
    pivot_table = monthly_returns.pivot(index="year", columns="month", values="return")
    fig, ax = plt.subplots(figsize=(12, max(6, len(pivot_table) * 0.5)))
    sns.heatmap(
        pivot_table,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        center=0,
        cbar_kws={"label": "Monthly Return (%)"},
        linewidths=0.5,
        ax=ax,
    )
    ax.set_xlabel("Month", fontsize=12, fontweight="bold")
    ax.set_ylabel("Year", fontsize=12, fontweight="bold")
    ax.set_title(f"{ticker} - Monthly Returns Heatmap (%)", fontsize=14, fontweight="bold")
    ax.set_xticklabels(
        ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    )
    plt.tight_layout()
    if config.SAVE_FIGURES:
        filename = f"monthly_returns_{ticker}_{config.START_DATE}_{config.END_DATE}.png"
        plt.savefig(filename, dpi=config.FIGURE_DPI, bbox_inches="tight")
        print(f"✓ Saved monthly returns heatmap to: {filename}")
    plt.show()
    plt.close()


def plot_trade_analysis(df, ticker, config):
    """
    Plot trade analysis including entry/exit points and holding period distribution.

    Visualizes actual trades on the price chart and analyzes trade durations.

    Parameters:
    -----------
    df : pd.DataFrame
        Backtest results
    ticker : str
        Ticker symbol
    config : StrategyConfig
        Configuration object
    """
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(config.FIGURE_SIZE[0], config.FIGURE_SIZE[1] * 1.2)
    )
    ax1.plot(df["date"], df["close"], label="Close Price", linewidth=1.5, alpha=0.7, color="blue")
    buys = df[df["trade"] == "BUY"]
    ax1.scatter(buys["date"], buys["open"], color="green", marker="^", s=100, label="Buy", zorder=5)
    sells = df[df["trade"] == "SELL"]
    ax1.scatter(
        sells["date"], sells["open"], color="red", marker="v", s=100, label="Sell", zorder=5
    )
    ax1.set_xlabel("Date", fontsize=12, fontweight="bold")
    ax1.set_ylabel("Price ($)", fontsize=12, fontweight="bold")
    ax1.set_title(f"{ticker} - Entry/Exit Points", fontsize=14, fontweight="bold")
    ax1.legend(loc="best", fontsize=10)
    ax1.grid(True, alpha=0.3)
    buy_dates = df[df["trade"] == "BUY"]["date"].reset_index(drop=True)
    sell_dates = df[df["trade"] == "SELL"]["date"].reset_index(drop=True)
    if len(buy_dates) > 0 and len(sell_dates) > 0:
        min_len = min(len(buy_dates), len(sell_dates))
        holding_periods = [(sell_dates.iloc[i] - buy_dates.iloc[i]).days for i in range(min_len)]
        ax2.hist(holding_periods, bins=20, alpha=0.7, color="steelblue", edgecolor="black")
        ax2.axvline(
            np.mean(holding_periods),
            color="red",
            linestyle="--",
            linewidth=2,
            label=f"Mean: {np.mean(holding_periods):.1f} days",
        )
        ax2.set_xlabel("Holding Period (days)", fontsize=12, fontweight="bold")
        ax2.set_ylabel("Frequency", fontsize=12, fontweight="bold")
        ax2.set_title(
            f"{ticker} - Trade Holding Period Distribution", fontsize=14, fontweight="bold"
        )
        ax2.legend(fontsize=10)
        ax2.grid(True, alpha=0.3)
    else:
        ax2.text(
            0.5,
            0.5,
            "No completed trades",
            ha="center",
            va="center",
            transform=ax2.transAxes,
            fontsize=14,
        )
        ax2.set_title(
            f"{ticker} - Trade Holding Period Distribution", fontsize=14, fontweight="bold"
        )
    plt.tight_layout()
    if config.SAVE_FIGURES:
        filename = f"trade_analysis_{ticker}_{config.START_DATE}_{config.END_DATE}.png"
        plt.savefig(filename, dpi=config.FIGURE_DPI, bbox_inches="tight")
        print(f"✓ Saved trade analysis to: {filename}")
    plt.show()
    plt.close()


def plot_performance_comparison(metrics_df, benchmark_metrics, config):
    """
    Plot performance comparison bar charts across all strategies.

    Compares key metrics (returns, Sharpe, drawdown) across all tickers and benchmark.

    Parameters:
    -----------
    metrics_df : pd.DataFrame
        DataFrame with metrics for all strategies
    benchmark_metrics : dict
        Benchmark metrics
    config : StrategyConfig
        Configuration object
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    if benchmark_metrics:
        comparison_df = pd.concat(
            [
                metrics_df[
                    [
                        "ticker",
                        "annualized_return_pct",
                        "sharpe_ratio",
                        "max_drawdown_pct",
                        "win_rate_pct",
                    ]
                ],
                pd.DataFrame(
                    [
                        {
                            "ticker": benchmark_metrics["ticker"],
                            "annualized_return_pct": benchmark_metrics["annualized_return_pct"],
                            "sharpe_ratio": benchmark_metrics["sharpe_ratio"],
                            "max_drawdown_pct": benchmark_metrics["max_drawdown_pct"],
                            "win_rate_pct": np.nan,
                        }
                    ]
                ),
            ],
            ignore_index=True,
        )
    else:
        comparison_df = metrics_df[
            ["ticker", "annualized_return_pct", "sharpe_ratio", "max_drawdown_pct", "win_rate_pct"]
        ].copy()
    ax1 = axes[0, 0]
    colors = ["green" if x > 0 else "red" for x in comparison_df["annualized_return_pct"]]
    ax1.barh(
        comparison_df["ticker"], comparison_df["annualized_return_pct"], color=colors, alpha=0.7
    )
    ax1.set_xlabel("Annualized Return (%)", fontsize=10, fontweight="bold")
    ax1.set_title("Annualized Returns Comparison", fontsize=12, fontweight="bold")
    ax1.grid(True, alpha=0.3, axis="x")
    ax1.axvline(x=0, color="black", linestyle="-", linewidth=0.8)
    ax2 = axes[0, 1]
    colors = [
        "green" if x > 1 else "orange" if x > 0 else "red" for x in comparison_df["sharpe_ratio"]
    ]
    ax2.barh(comparison_df["ticker"], comparison_df["sharpe_ratio"], color=colors, alpha=0.7)
    ax2.set_xlabel("Sharpe Ratio", fontsize=10, fontweight="bold")
    ax2.set_title("Sharpe Ratio Comparison", fontsize=12, fontweight="bold")
    ax2.grid(True, alpha=0.3, axis="x")
    ax2.axvline(x=0, color="black", linestyle="-", linewidth=0.8)
    ax2.axvline(x=1, color="blue", linestyle="--", linewidth=1, alpha=0.5)
    ax3 = axes[1, 0]
    colors = [
        "red" if x < -20 else "orange" if x < -10 else "green"
        for x in comparison_df["max_drawdown_pct"]
    ]
    ax3.barh(comparison_df["ticker"], comparison_df["max_drawdown_pct"], color=colors, alpha=0.7)
    ax3.set_xlabel("Maximum Drawdown (%)", fontsize=10, fontweight="bold")
    ax3.set_title("Maximum Drawdown Comparison", fontsize=12, fontweight="bold")
    ax3.grid(True, alpha=0.3, axis="x")
    ax3.axvline(x=0, color="black", linestyle="-", linewidth=0.8)
    ax4 = axes[1, 1]
    strategy_df = comparison_df[comparison_df["win_rate_pct"].notna()]
    if len(strategy_df) > 0:
        colors = ["green" if x > 50 else "red" for x in strategy_df["win_rate_pct"]]
        ax4.barh(strategy_df["ticker"], strategy_df["win_rate_pct"], color=colors, alpha=0.7)
        ax4.set_xlabel("Win Rate (%)", fontsize=10, fontweight="bold")
        ax4.set_title("Win Rate Comparison", fontsize=12, fontweight="bold")
        ax4.grid(True, alpha=0.3, axis="x")
        ax4.axvline(x=50, color="blue", linestyle="--", linewidth=1, alpha=0.5)
    else:
        ax4.text(
            0.5,
            0.5,
            "No trade data",
            ha="center",
            va="center",
            transform=ax4.transAxes,
            fontsize=14,
        )
    plt.tight_layout()
    if config.SAVE_FIGURES:
        filename = f"performance_comparison_{config.START_DATE}_{config.END_DATE}.png"
        plt.savefig(filename, dpi=config.FIGURE_DPI, bbox_inches="tight")
        print(f"✓ Saved performance comparison to: {filename}")
    plt.show()
    plt.close()


def plot_walk_forward_results(wf_results, config):
    """
    Plot walk-forward analysis results showing out-of-sample consistency.

    Visualizes year-by-year performance to assess strategy robustness.

    Parameters:
    -----------
    wf_results : list
        List of walk-forward results
    config : StrategyConfig
        Configuration object
    """
    if len(wf_results) == 0:
        print("No walk-forward results to plot")
        return
    wf_df = pd.DataFrame(wf_results)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=config.FIGURE_SIZE)
    tickers = wf_df["ticker"].unique()
    for ticker in tickers:
        ticker_data = wf_df[wf_df["ticker"] == ticker]
        ax1.plot(
            ticker_data["period"],
            ticker_data["annualized_return_pct"],
            marker="o",
            label=ticker,
            linewidth=2,
        )
    ax1.axhline(y=0, color="black", linestyle="--", linewidth=1)
    ax1.set_xlabel("Period", fontsize=12, fontweight="bold")
    ax1.set_ylabel("Annualized Return (%)", fontsize=12, fontweight="bold")
    ax1.set_title("Walk-Forward: Annual Returns by Period", fontsize=14, fontweight="bold")
    ax1.legend(loc="best", fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis="x", rotation=45)
    for ticker in tickers:
        ticker_data = wf_df[wf_df["ticker"] == ticker]
        ax2.plot(
            ticker_data["period"],
            ticker_data["sharpe_ratio"],
            marker="s",
            label=ticker,
            linewidth=2,
        )
    ax2.axhline(y=0, color="black", linestyle="--", linewidth=1)
    ax2.axhline(y=1, color="blue", linestyle="--", linewidth=1, alpha=0.5)
    ax2.set_xlabel("Period", fontsize=12, fontweight="bold")
    ax2.set_ylabel("Sharpe Ratio", fontsize=12, fontweight="bold")
    ax2.set_title("Walk-Forward: Sharpe Ratio by Period", fontsize=14, fontweight="bold")
    ax2.legend(loc="best", fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    if config.SAVE_FIGURES:
        filename = f"walk_forward_analysis_{config.START_DATE}_{config.END_DATE}.png"
        plt.savefig(filename, dpi=config.FIGURE_DPI, bbox_inches="tight")
        print(f"✓ Saved walk-forward analysis to: {filename}")
    plt.show()
    plt.close()


def generate_all_visualizations(results_dict, benchmark_df, all_metrics, wf_results, config):
    """
    Generate all visualization plots for the strategy.

    Master function that creates comprehensive visual analysis including:
    - Equity curves
    - Drawdown charts
    - Returns distributions
    - Monthly returns heatmaps
    - Trade analysis
    - Performance comparison
    - Walk-forward analysis

    Parameters:
    -----------
    results_dict : dict
        Dictionary with ticker as key and backtest DataFrame as value
    benchmark_df : pd.DataFrame
        Benchmark data
    all_metrics : list
        List of metrics dictionaries
    wf_results : list
        Walk-forward results
    config : StrategyConfig
        Configuration object
    """
    print("\n" + "=" * 80)
    print("GENERATING VISUALIZATIONS")
    print("=" * 80)
    print("→ Creating equity curve comparison...")
    plot_equity_curve(results_dict, benchmark_df, config)
    print("→ Creating drawdown analysis...")
    plot_drawdown(results_dict, config)
    print("→ Creating returns distribution...")
    plot_returns_distribution(results_dict, config)
    print("→ Creating monthly returns heatmaps...")
    for ticker, df in results_dict.items():
        plot_monthly_returns_heatmap(df, ticker, config)
    print("→ Creating trade analysis...")
    for ticker, df in results_dict.items():
        plot_trade_analysis(df, ticker, config)
    print("→ Creating performance comparison...")
    metrics_df = pd.DataFrame(all_metrics)
    benchmark_metrics = (
        calculate_benchmark_metrics(benchmark_df, config) if benchmark_df is not None else None
    )
    plot_performance_comparison(metrics_df, benchmark_metrics, config)
    if len(wf_results) > 0:
        print("→ Creating walk-forward analysis...")
        plot_walk_forward_results(wf_results, config)
    print("=" * 80)
    print("VISUALIZATION COMPLETE")
    print("=" * 80)


def print_performance_report(metrics, benchmark_metrics, wf_results):
    """
    Print comprehensive performance report.

    Parameters:
    -----------
    metrics : dict
        Strategy performance metrics
    benchmark_metrics : dict
        Benchmark performance metrics
    wf_results : list
        Walk-forward analysis results
    """
    print("\n" + "=" * 80)
    print("PERFORMANCE REPORT")
    print("=" * 80)
    print(f"\n{metrics['ticker']} - FULL PERIOD RESULTS")
    print("-" * 80)
    print(f"Total Return:              {metrics['total_return_pct']:>10.2f}%")
    print(f"Annualized Return:         {metrics['annualized_return_pct']:>10.2f}%")
    print(f"Volatility (Annual):       {metrics['volatility_pct']:>10.2f}%")
    print(f"Sharpe Ratio:              {metrics['sharpe_ratio']:>10.2f}")
    print(f"Sortino Ratio:             {metrics['sortino_ratio']:>10.2f}")
    print(f"Maximum Drawdown:          {metrics['max_drawdown_pct']:>10.2f}%")
    print(f"Calmar Ratio:              {metrics['calmar_ratio']:>10.2f}")
    print(f"Win Rate:                  {metrics['win_rate_pct']:>10.2f}%")
    print(f"Profit Factor:             {metrics['profit_factor']:>10.2f}")
    print(f"Number of Trades:          {metrics['num_trades']:>10.0f}")
    print(f"Avg Trade P&L:             ${metrics['avg_trade_pnl']:>10.2f}")
    print(f"Final Portfolio Value:     ${metrics['final_portfolio_value']:>10,.2f}")
    print("\nOVERFITTING TESTS:")
    print(f"Probabilistic Sharpe:      {metrics['psr']:>10.2%} (target: >95%)")
    print(f"Deflated Sharpe Ratio:     {metrics['dsr']:>10.2f} (target: >0)")
    if benchmark_metrics:
        print(f"\n{benchmark_metrics['ticker']} - BENCHMARK (BUY & HOLD)")
        print("-" * 80)
        print(f"Total Return:              {benchmark_metrics['total_return_pct']:>10.2f}%")
        print(f"Annualized Return:         {benchmark_metrics['annualized_return_pct']:>10.2f}%")
        print(f"Volatility (Annual):       {benchmark_metrics['volatility_pct']:>10.2f}%")
        print(f"Sharpe Ratio:              {benchmark_metrics['sharpe_ratio']:>10.2f}")
        print(f"Maximum Drawdown:          {benchmark_metrics['max_drawdown_pct']:>10.2f}%")
        print(f"\nRELATIVE PERFORMANCE vs {benchmark_metrics['ticker']}")
        print("-" * 80)
        excess_return = (
            metrics["annualized_return_pct"] - benchmark_metrics["annualized_return_pct"]
        )
        print(f"Excess Return (Annual):    {excess_return:>10.2f}%")
        sharpe_diff = metrics["sharpe_ratio"] - benchmark_metrics["sharpe_ratio"]
        print(f"Sharpe Difference:         {sharpe_diff:>10.2f}")
    if len(wf_results) > 0:
        print("\nWALK-FORWARD ANALYSIS (Expanding Window)")
        print("-" * 80)
        wf_df = pd.DataFrame(wf_results)
        print(
            wf_df[
                [
                    "period",
                    "annualized_return_pct",
                    "sharpe_ratio",
                    "max_drawdown_pct",
                    "num_trades",
                ]
            ].to_string(index=False)
        )
        print("\nWALK-FORWARD CONSISTENCY METRICS:")
        print(f"Avg Annual Return:         {wf_df['annualized_return_pct'].mean():>10.2f}%")
        print(f"Std Annual Return:         {wf_df['annualized_return_pct'].std():>10.2f}%")
        print(f"Avg Sharpe Ratio:          {wf_df['sharpe_ratio'].mean():>10.2f}")
        print(
            f"Positive Years:            {(wf_df['annualized_return_pct'] > 0).sum()}/{len(wf_df)}"
        )
    print("=" * 80)


def save_results(df, metrics, ticker, config):
    """
    Save backtest results and metrics to CSV files.

    Parameters:
    -----------
    df : pd.DataFrame
        Backtest results
    metrics : dict
        Performance metrics
    ticker : str
        Ticker symbol
    config : StrategyConfig
        Configuration object
    """
    output_filename = f"connors_mdd_{ticker}_backtest_{config.START_DATE}_{config.END_DATE}.csv"
    df.to_csv(output_filename, index=False)
    print(f"\n✓ Saved backtest results to: {output_filename}")
    metrics_filename = f"connors_mdd_{ticker}_metrics_{config.START_DATE}_{config.END_DATE}.csv"
    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(metrics_filename, index=False)
    print(f"✓ Saved performance metrics to: {metrics_filename}")


def run_strategy():
    """
    Main execution function for Larry Connors MDU/MDD strategy.

    WORKFLOW:
    1. Load configuration
    2. Download data from Yahoo Finance
    3. Calculate technical features (with T-1 lag)
    4. Generate trading signals
    5. Execute backtest with realistic costs
    6. Calculate performance metrics
    7. Run walk-forward analysis
    8. Generate comparison with benchmark
    9. Generate comprehensive visualizations (NEW)
    10. Print comprehensive report
    11. Save results to CSV
    """
    config = StrategyConfig()
    print("\n" + "=" * 80)
    print("LARRY CONNORS MDU/MDD MEAN REVERSION STRATEGY")
    print("Based on: High Probability Trading, Chapter 6")
    print("=" * 80)
    print("\nSTRATEGY RULES:")
    print(f"✓ Entry: Price > {config.MA_LONG_PERIOD}MA AND Price < {config.MA_SHORT_PERIOD}MA")
    print(f"✓ Entry: {config.MIN_DOWN_DAYS} down days out of last {config.LOOKBACK_DAYS} days")
    print(f"✓ Exit: Price > {config.MA_SHORT_PERIOD}MA")
    print("✓ No stop-loss (per original Connors strategy)")
    print("\nEXECUTION:")
    print("✓ Signal at T-1 close → Execute at T open")
    print("✓ No look-ahead bias")
    print(f"✓ Commission: {config.COMMISSION_PCT * 100}% per trade")
    print(f"✓ Position sizing: {('Compounding' if config.COMPOUNDING_MODE else 'Fixed Notional')}")
    print("=" * 80)
    all_data = load_all_data(config)
    if len(all_data) == 0:
        print("\nERROR: No data available. Cannot proceed with backtest.")
        print("Please check:")
        print("1. Internet connection")
        print("2. Yahoo Finance key validity")
        print("3. Ticker symbols are correct")
        return
    all_metrics = []
    all_wf_results = []
    results_dict = {}
    valid_tickers = [t for t in config.TICKERS if t and len(t) > 0]
    for ticker in valid_tickers:
        if ticker not in all_data:
            print(f"\nSkipping {ticker} - no data available")
            continue
        print(f"\n{'=' * 80}")
        print(f"PROCESSING: {ticker}")
        print(f"{'=' * 80}")
        df = all_data[ticker].copy()
        print("→ Calculating technical features...")
        df = calculate_features(df, config)
        print("→ Generating trading signals...")
        df = generate_signals(df, config)
        print("→ Running backtest...")
        df = backtest_strategy(df, config, ticker)
        results_dict[ticker] = df
        print("→ Calculating performance metrics...")
        metrics = calculate_performance_metrics(df, config, ticker)
        all_metrics.append(metrics)
        print("→ Running walk-forward analysis...")
        wf_results = walk_forward_analysis(df, config, ticker)
        all_wf_results.extend(wf_results)
        save_results(df, metrics, ticker, config)
    benchmark_metrics = None
    benchmark_df = None
    if config.BENCHMARK_TICKER in all_data:
        print(f"\n{'=' * 80}")
        print(f"PROCESSING BENCHMARK: {config.BENCHMARK_TICKER}")
        print(f"{'=' * 80}")
        benchmark_df = all_data[config.BENCHMARK_TICKER].copy()
        benchmark_metrics = calculate_benchmark_metrics(benchmark_df, config)
    if len(results_dict) > 0:
        generate_all_visualizations(results_dict, benchmark_df, all_metrics, all_wf_results, config)
    for metrics in all_metrics:
        wf_results_ticker = [r for r in all_wf_results if r["ticker"] == metrics["ticker"]]
        print_performance_report(metrics, benchmark_metrics, wf_results_ticker)
    if len(all_metrics) > 0:
        combined_metrics = pd.DataFrame(all_metrics)
        combined_filename = (
            f"connors_mdd_combined_metrics_{config.START_DATE}_{config.END_DATE}.csv"
        )
        combined_metrics.to_csv(combined_filename, index=False)
        print(f"\n✓ Saved combined metrics to: {combined_filename}")
    if len(all_wf_results) > 0:
        wf_df = pd.DataFrame(all_wf_results)
        wf_filename = f"connors_mdd_walkforward_{config.START_DATE}_{config.END_DATE}.csv"
        wf_df.to_csv(wf_filename, index=False)
        print(f"✓ Saved walk-forward results to: {wf_filename}")
    print("\n" + "=" * 80)
    print("BACKTEST COMPLETE")
    print("=" * 80)


if __name__ == "__main__":
    run_strategy()
